# DataFrame Input Test

This notebook tests that `jscatter` works with five different data sources:

1. **NumPy** arrays (vanilla)
2. **Pandas** DataFrame
3. **Polars** DataFrame
4. **PyArrow** Table
5. **DuckDB** query result

All five should produce identical scatter plots.

In [1]:
import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import duckdb
import jscatter

In [2]:
# Shared data
np.random.seed(42)
n = 1000

x = np.random.randn(n)
y = np.random.randn(n)
value = np.sqrt(x**2 + y**2)
group = np.array(['A', 'B', 'C', 'D'])[np.random.randint(0, 4, n)]

## 1. NumPy Arrays

In [3]:
scatter_np = jscatter.Scatter(x=x, y=y, color_by=value, legend=True)
scatter_np.show()

## 2. Pandas DataFrame

In [4]:
df_pandas = pd.DataFrame(
    {'x': x, 'y': y, 'value': value, 'group': pd.Categorical(group)}
)

scatter_pd = jscatter.Scatter(
    data=df_pandas, x='x', y='y', color_by='group', size_by='value', legend=True
)
scatter_pd.show()

## 3. Polars DataFrame

In [5]:
df_polars = pl.DataFrame({'x': x, 'y': y, 'value': value, 'group': group}).cast(
    {'group': pl.Categorical}
)

scatter_pl = jscatter.Scatter(
    data=df_polars, x='x', y='y', color_by='group', size_by='value', legend=True
)
scatter_pl.show()

## 4. PyArrow Table

In [6]:
table_arrow = pa.table(
    {
        'x': x,
        'y': y,
        'value': value,
        'group': pa.DictionaryArray.from_arrays(
            pa.array(np.searchsorted(np.unique(group), group), type=pa.int32()),
            pa.array(np.unique(group)),
        ),
    }
)

scatter_pa = jscatter.Scatter(
    data=table_arrow, x='x', y='y', color_by='group', size_by='value', legend=True
)
scatter_pa.show()

## 5. DuckDB Query Result

In [7]:
conn = duckdb.connect()
conn.execute('CREATE TABLE points AS SELECT * FROM df_pandas')
table_duckdb = conn.sql('SELECT * FROM points').arrow()

scatter_db = jscatter.Scatter(
    data=table_duckdb, x='x', y='y', color_by='group', size_by='value', legend=True
)
scatter_db.show()

## Selection Roundtrip

Verify that selections from a Polars-backed scatter return the same rows as from a Pandas-backed scatter, and that the integer indices can index directly back into the original Polars DataFrame (no `.tolist()` needed).

In [8]:
indices = [0, 10, 50, 100]

# Select the same points in both Pandas and Polars scatters
scatter_pd.selection(indices)
scatter_pl.selection(indices)

# Get selections back
sel_pd = scatter_pd.selection()
sel_pl = scatter_pl.selection()

print('Pandas selection: ', sel_pd)
print('Polars selection: ', sel_pl)
print('Identical:        ', np.array_equal(sel_pd, sel_pl))

# Index back into the original DataFrames
rows_pd = df_pandas.iloc[sel_pd]
rows_pl = df_polars[sel_pl]  # numpy array works directly, no .tolist() needed

print('\nPandas rows:')
print(rows_pd[['x', 'y', 'group']].to_string(index=False))

print('\nPolars rows:')
print(rows_pl.select('x', 'y', 'group'))

Pandas selection:  [  0  10  50 100]
Polars selection:  [  0  10  50 100]
Identical:         True

Pandas rows:
        x         y group
 0.496714  1.399355     B
-0.463418  1.317394     A
 0.324084 -0.625563     A
-1.415371  0.998010     B

Polars rows:
shape: (4, 3)
┌───────────┬───────────┬───────┐
│ x         ┆ y         ┆ group │
│ ---       ┆ ---       ┆ ---   │
│ f64       ┆ f64       ┆ cat   │
╞═══════════╪═══════════╪═══════╡
│ 0.496714  ┆ 1.399355  ┆ B     │
│ -0.463418 ┆ 1.317394  ┆ A     │
│ 0.324084  ┆ -0.625563 ┆ A     │
│ -1.415371 ┆ 0.99801   ┆ B     │
└───────────┴───────────┴───────┘


## Data Update Across Types

Swap data from one format to another on an existing scatter.

In [11]:
# Start with a Pandas DF scatter
scatter_swap = jscatter.Scatter(data=df_pandas, x='x', y='y', color_by='group')
scatter_swap.show()

In [12]:
# Now swap in a Polars DataFrame -- the scatter above should update
new_polars = pl.DataFrame(
    {
        'x': np.random.randn(500).tolist(),
        'y': np.random.randn(500).tolist(),
        'group': (['X', 'Y'] * 250),
    }
).cast({'group': pl.Categorical})

scatter_swap.data(new_polars)
scatter_swap.color(by='group')